In [25]:
import time
import random
import threading
from IPython.display import display, clear_output

def selection_sort(arr, progress_callback=None):
    n = len(arr)
    for i in range(n):
        min_idx = i    # 設最小值
        for j in range(i + 1, n):     # 找最小的元素並放到最小值
            if arr[j] < arr[min_idx]:
                min_idx = j
        arr[i], arr[min_idx] = arr[min_idx], arr[i]
        if progress_callback and n > 0:
            if n > 100:
                if i % (n // 100) == 0 or i == n - 1:      # 顯示進度
                    progress_callback((i + 1) / n)
            else:
                progress_callback((i + 1) / n)
    return arr

def bubble_sort(arr, progress_callback=None):
    n = len(arr)
    for i in range(n):        # 外圈從最後面開始往前找值
        for j in range(0, n - i - 1):      # 如果前面的值比較大就交換
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
        if progress_callback and n > 0:
            if n > 100:
                if i % (n // 100) == 0 or i == n - 1:      # 顯示進度
                    progress_callback((i + 1) / n)
            else:
                progress_callback((i + 1) / n)
    return arr

def insertion_sort(arr, progress_callback=None):
    n = len(arr)
    for i in range(1, n):
        key = arr[i]
        j = i - 1
        while j >= 0 and key < arr[j]:    # 找到以排序的最後一個並回頭將key插入比自己小的位置
            arr[j + 1] = arr[j]
            j -= 1
        arr[j + 1] = key
        if progress_callback and n > 0:
            if n > 100:
                if i % (n // 100) == 0 or i == n - 1:      # 顯示進度
                    progress_callback((i + 1) / n)
            else:
                progress_callback((i + 1) / n)
    return arr

def quick_sort(arr, start, end):
    if start >= end:
        return
    pivot = start
    left = start
    right = end

    while left != right:
        while arr[right] >= arr[pivot] and left != right:     # 交換中心點左右順序錯誤的數字
            right -= 1
        while arr[left] <= arr[pivot] and left != right:
            left += 1
        arr[left], arr[right] = arr[right], arr[left]


    arr[pivot], arr[right] = arr[right], arr[pivot]        # 將中心點放到正確位置
    pivot = right

    quick_sort(arr, start, pivot - 1)        # 排序左右兩邊的數字
    quick_sort(arr, pivot + 1, end)

# 畫進度條
progress_display_objects = {}
bar_length = 20
def get_progress_bar_string(label, progress_percent, final_duration_str="", speed_status="Measuring..."):
    filled_length = int(bar_length * progress_percent)
    bar_segment = '#' * filled_length + '-' * (bar_length - filled_length)
    if final_duration_str:
        return f"{label}: |{bar_segment}| {int(progress_percent * 100)}%  time: {final_duration_str}"
    else:
        return f"{label}: |{bar_segment}| {int(progress_percent * 100)}%  time: "

# 顯示運行進度並計時
def measure_sort_time(sort_func, arr_copy, label, results_dict):
    def progress_callback(progress_ratio):        # 即時更新進度條
        progress_display_objects[label].update(get_progress_bar_string(label, progress_ratio))

    actual_start_sort_time = time.time()

    if sort_func.__name__ in ['selection_sort', 'bubble_sort', 'insertion_sort']:
        sort_func(arr_copy, progress_callback)
    else:
        sort_func(arr_copy, 0, len(arr_copy) - 1)        # quick_sort的顯示（太快了不好顯示）
        progress_display_objects[label].update(get_progress_bar_string(label, 1.0))

    actual_end_sort_time = time.time()
    duration = actual_end_sort_time - actual_start_sort_time    # 計時

    progress_display_objects[label].update(get_progress_bar_string(label, 1.0, f"{duration:.2f}s", "Measuring..."))
    results_dict[label] = duration


while True:
    try:
        list_size = int(input("請輸入要產生的資料筆數："))     # 輸入資料數量
        if list_size > 0:
            break
        print("請輸入大於 0 的整數！")
    except ValueError:
        print("請輸入整數！")

random_list = [random.randint(0, list_size *2 ) for _ in range(list_size)]    # 建立5000個隨機整數

sort_functions = {
    'Selection Sort': selection_sort,        # 建立儲存空間
    'Bubble Sort': bubble_sort,
    'Insertion Sort': insertion_sort,
    'Quick Sort': quick_sort
}

# 建立初始進度條
for name in sort_functions.keys():
    progress_display_objects[name] = display(get_progress_bar_string(name, 0), display_id=name)

# 建立執行緒和存結果的字典
threads = []
thread_results = {}
for name, func in sort_functions.items():
    thread = threading.Thread(target=measure_sort_time, args=(func, list(random_list), name, thread_results))
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

# The results are now in thread_results
raw_sort_results = thread_results

# 清除輸出
progress_display_objects = {}

請輸入要產生的資料筆數：10000


'Selection Sort: |####################| 100%  time: 9.62s'

'Bubble Sort: |####################| 100%  time: 12.90s'

'Insertion Sort: |####################| 100%  time: 11.21s'

'Quick Sort: |####################| 100%  time: 0.17s'